# STAGE 11 · Classifier-Conditioned Report Generation

## The measured gap this exploits

| | clinical F1 |
|---|---|
| current report generator | 0.5799 |
| **ceiling if it simply stated the classifier** | **0.6535** |
| headroom | **+0.0736** |
| *(oracle, with perfect labels)* | *0.9203* |

Your classifier reaches mean AUROC **0.8554** (cardiomegaly 0.9189). Your report generator's entire margin over a fixed-string baseline is **+0.0149** — it is barely reading the image. So stop asking the decoder to rediscover pathology from pixels and tell it what the classifier found.

**Why this differs from Stage 10, which failed:** there the classifier already had the projection information, so conditioning gained +0.0003. Here there is a *measured* 0.0736 of information the decoder is not using.

## How

```
positive: cardiomegaly, pleural effusion. negative: edema, pneumothorax.
```

That text is embedded with **BART's own embedding table** and prepended to the 144 visual tokens. **Zero new parameters** — so `best.pt` loads with 0 missing keys and an empty prompt is bit-identical to Stage 4.

| decision | why |
|---|---|
| train on classifier **predictions**, not ground truth | training on a perfect oracle it never sees at inference fails exactly when the classifier is wrong |
| **15% label dropout** | without it the decoder transcribes the prompt and stops reading the image |
| targets stay Stage-1 cleaned | prior hallucination is 0.0000 and must stay there |

## 🔒 Both checkpoints are read-only

`checkpoints/stage4/best.pt` and `checkpoints/stage5/best.pt` are SHA-256 verified before and after. All output goes to `checkpoints/stage11/`. A hard `assert` aborts if any output path resolves under either.

---
# 0 · Config & safety

**Run `SMOKE_TEST = True` first.** ~6 min, ~0.2 CU.

In [ ]:
# rouge-score is not preinstalled on Colab. Installing here rather than at
# section 4 means a missing dependency fails in the first 20 seconds instead
# of after the tar has been staged and the model loaded.
import subprocess, sys as _s
try:
    import rouge_score  # noqa: F401
except ImportError:
    print('  installing rouge-score ...')
    subprocess.run([_s.executable, '-m', 'pip', 'install', '-q', 'rouge-score'], check=True)
    import rouge_score  # noqa: F401
print('  rouge-score ready')

import os, sys, json, time, math, hashlib, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, torch

SMOKE_TEST   = True      # <-- True first. ALWAYS.
EPOCHS       = 3
LR_VISION    = 5e-6      # trunk is already converged; nudge only
LR_REST      = 5e-5      # proj + BART, 10x -- matches Stage 4's PROJ_LR_MULT
PROMPT_DROP  = 0.15
WEIGHT_DECAY, GRAD_CLIP, NUM_WORKERS = 0.01, 1.0, 2
MAX_TOKENS, GEN_MAX, GEN_MIN = 256, 192, 24

# Stage 4 reference, from its full test run. The bar Stage 11 must clear.
S4 = dict(rougeL=0.2918, margin=0.0149, clinical_f1=0.5799, prior=0.0000,
          gen_words=36.62, baseline=0.2769)

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
else: print('  Drive already mounted')
PROJECT  = Path('/content/drive/MyDrive/Component_01')
IMG_ROOT = Path('/content/cardio_image_384')
TAR      = PROJECT / 'data' / 'images' / 'cardio_384.tar'
MANIFEST = PROJECT / 'training_manifest'
S4_CKPT  = PROJECT / 'checkpoints' / 'stage4' / 'best.pt'
S5_CKPT  = PROJECT / 'checkpoints' / 'stage5' / 'best.pt'
S6CACHE  = PROJECT / 'reports' / 'stage6' / 'cache'
CKPT_DIR = PROJECT / 'checkpoints' / 'stage11'; CKPT_DIR.mkdir(parents=True, exist_ok=True)
OUT      = PROJECT / 'reports' / 'stage11'; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT))

# ---- HARD SAFETY GUARDS ----
PROT = [(PROJECT/'checkpoints'/'stage4').resolve(), (PROJECT/'checkpoints'/'stage5').resolve()]
for p in (CKPT_DIR, OUT):
    r = p.resolve()
    for q in PROT:
        assert r != q and q not in r.parents, 'output collides with a protected checkpoint dir!'
SHA = {n: hashlib.sha256(p.read_bytes()).hexdigest() for n, p in
       (('stage4', S4_CKPT), ('stage5', S5_CKPT))}
for n, h in SHA.items(): print('  %s best.pt sha %s' % (n, h[:40]))
print('  outputs ->', CKPT_DIR)

DEV = 'cuda'; assert torch.cuda.is_available(), 'select an L4 GPU'
import subprocess
print(' ', subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
      '--format=csv,noheader'], capture_output=True, text=True).stdout.strip())
print('  mode:', 'SMOKE TEST' if SMOKE_TEST else 'FULL RUN')

---
# 0b · Stage the images

In [ ]:
if not IMG_ROOT.exists():
    import shutil
    t0 = time.time(); lt = Path('/content/cardio_384.tar')
    if not lt.exists():
        assert TAR.exists(), 'tar not found: ' + str(TAR)
        print('  copying %.1f GB off Drive...' % (TAR.stat().st_size / 1e9))
        shutil.copy(TAR, lt)
    subprocess.run(['tar', '-xf', str(lt), '-C', '/content'], check=True)
    print('  staged in %.1f min' % ((time.time() - t0) / 60))
    try: lt.unlink()
    except OSError: pass
if not IMG_ROOT.exists():
    cand = [p for p in Path('/content').glob('*') if p.is_dir() and (p/'test').is_dir()]
    assert cand, 'extraction produced no directory containing test/'
    IMG_ROOT = cand[0]
print('  images:', IMG_ROOT, IMG_ROOT.exists())

---
# 1 · Gate: self-tests

**128 tests across five modules.**

In [ ]:
import stage6_acr as acr, stage9_fairness as s9, stage9b_gradrev as s9b
import stage10_conditional as s10, stage11_conditioned as s11
tot = 0
for nm, mod in (('stage6_acr', acr), ('stage9_fairness', s9), ('stage9b_gradrev', s9b),
                ('stage10_conditional', s10), ('stage11_conditioned', s11)):
    p, f = mod._selftest(verbose=False)
    print('  %-22s %3d passed  %d failed' % (nm, p, f))
    assert f == 0, nm + ' FAILED'
    tot += p
print('\n  ALL %d TESTS PASSED' % tot)

---
# 2 · Classifier probabilities for every split

val and test are already cached from Stage 6. **train is not** — the prompt needs it, so it is computed here once (~3 min on L4) and cached to Drive.

In [ ]:
from cxr_transforms import build_transform
from torch.utils.data import Dataset, DataLoader
PATH = s11.PATHOLOGIES
man = {s: pd.read_csv(MANIFEST / ('manifest_' + s + '.csv'), low_memory=False)
       for s in ('train', 'val', 'test')}
if SMOKE_TEST:
    man = {s: d.head(320).copy().reset_index(drop=True) for s, d in man.items()}
TF_EV = build_transform('test')

def clf_probs(split):
    f = S6CACHE / ('probs_' + split + '.npy')
    if not SMOKE_TEST and f.exists():
        X = np.load(f)
        if len(X) == len(man[split]):
            print('  %-5s cached (%d)' % (split, len(X))); return X
    m = s9b.CXRGradRev(len(PATH))
    ck = torch.load(S5_CKPT, map_location='cpu', weights_only=False)
    m.load_stage5(ck, use_ema=True); del ck
    m = m.eval().to(DEV).to(memory_format=torch.channels_last)
    ds = s9b.CXRDataset(man[split], IMG_ROOT, TF_EV, PATH)
    dl = DataLoader(ds, batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
    out, t0 = [], time.time()
    with torch.no_grad():
        for x, _, _, _ in dl:
            x = x.to(DEV, non_blocking=True).to(memory_format=torch.channels_last)
            with torch.autocast('cuda', dtype=torch.bfloat16):
                d, _ = m(x, None, lambd=0.0)
            out.append(torch.sigmoid(d.float()).cpu().numpy())
            n = sum(len(o) for o in out)
            print('\r  %-5s %d/%d' % (split, n, len(ds)), end='')
    X = np.concatenate(out).astype(np.float32)
    if not SMOKE_TEST: np.save(f, X)
    print('\r  %-5s %d in %.1f min' % (split, len(X), (time.time()-t0)/60))
    del m; torch.cuda.empty_cache()
    return X

PR = {s: clf_probs(s) for s in ('train', 'val', 'test')}
LV = man['val'][PATH].astype(int)
THR = [s9.best_f1_threshold(LV[k].values, PR['val'][:, j]) for j, k in enumerate(PATH)]
print('\n  thresholds:', [round(t, 3) for t in THR])
print('  example prompt:', s11.build_prompt(PR['test'][0], THR))

---
# 3 · Data & model

In [ ]:
from transformers import AutoTokenizer
from PIL import Image
TOK = AutoTokenizer.from_pretrained('GanjinZero/biobart-v2-base')
TF_TR = build_transform('train')

class RepDS(Dataset):
    def __init__(self, df, probs, thr, tf, dropout=0.0, seed=0):
        self.df, self.probs, self.thr = df.reset_index(drop=True), probs, thr
        self.tf, self.dropout = tf, dropout
        self.rng = np.random.default_rng(seed)
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        img = self.tf(Image.open(IMG_ROOT / self.df.image_path[i]))
        pr  = s11.build_prompt(self.probs[i], self.thr, dropout=self.dropout, rng=self.rng)
        ids = TOK(str(self.df.report[i]), truncation=True, max_length=MAX_TOKENS,
                  return_tensors='pt')['input_ids'][0]
        return img, ids, pr, str(self.df.report[i])

def collate(batch):
    imgs, seqs, prompts, texts = zip(*batch)
    n = max(len(s) for s in seqs)
    lab = torch.full((len(seqs), n), -100, dtype=torch.long)
    for i, s in enumerate(seqs): lab[i, :len(s)] = s
    pid, pmask = s11.encode_prompts(prompts, TOK)
    return torch.stack(imgs), lab, pid, pmask, list(texts)

dl_tr = DataLoader(RepDS(man['train'], PR['train'], THR, TF_TR, PROMPT_DROP),
                   batch_size=8, shuffle=True, num_workers=NUM_WORKERS,
                   collate_fn=collate, drop_last=True, pin_memory=True)
dl_va = DataLoader(RepDS(man['val'], PR['val'], THR, TF_EV, 0.0),
                   batch_size=16, shuffle=False, num_workers=NUM_WORKERS,
                   collate_fn=collate, pin_memory=True)
dl_te = DataLoader(RepDS(man['test'], PR['test'], THR, TF_EV, 0.0),
                   batch_size=16, shuffle=False, num_workers=NUM_WORKERS,
                   collate_fn=collate, pin_memory=True)
print('  batches  train %d  val %d  test %d' % (len(dl_tr), len(dl_va), len(dl_te)))

model = s11.CXRConditionedGenerator('GanjinZero/biobart-v2-base', 0.1, 1)
ck = torch.load(S4_CKPT, map_location='cpu', weights_only=False)
print('  load_stage4:', model.load_stage4(ck, use_ema=True))
print('  from epoch %d, val ROUGE-L %.4f' % (ck['epoch'], ck['best_metric']))
del ck
model = model.to(DEV)
STEPS = max(len(dl_tr), 1) * EPOCHS
opt = torch.optim.AdamW(model.param_groups(LR_VISION, LR_REST), weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(STEPS, 1))
print('  %d steps over %d epochs' % (STEPS, EPOCHS))

---
# 4 · Metrics

In [ ]:
import re
from rouge_score import rouge_scorer
_SC = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
PRIOR_RE = re.compile(r'(as compared (to|with)|in compar(ison|ed) (to|with)'
  r'|\b(prior|previous|earlier|preceding)\s+(studies|study|exams?|radiographs?|films?|imaging)'
  r'|\b(unchanged|stable|constant)\b|\bno (significant |relevant |interval )?changes?\b'
  r'|\b(increasing|decreasing|worsening|improving)\b|___'
  r'|\b(again|persistent|persists|remains?)\b)', re.I)
KW = {'Cardiomegaly': r'(cardiomegaly|cardiac enlargement|enlarged cardiac silhouette|heart.{0,20}enlarged)',
      'Edema': r'(pulmonary edema|interstitial edema|\bedema\b|vascular congestion)',
      'Pleural_Effusion': r'(pleural effusion|\beffusions?\b)', 'Atelectasis': r'atelecta',
      'Consolidation': r'consolidat', 'Lung_Opacity': r'(opacit|infiltrate)',
      'Pneumonia': r'pneumonia', 'Pneumothorax': r'pneumothora'}
NEG = re.compile(r'\b(no|not|without|negative for|free of|absence of|absent)\b', re.I)
def assert_labels(t):
    o = {}; ss = re.split(r'(?<=[.;])\s+', re.sub(r'\s+', ' ', t or ''))
    for k, p in KW.items():
        kw = re.compile(p, re.I); v = 0
        for s in ss:
            for m_ in kw.finditer(s):
                if not NEG.search(s[:m_.start()]): v = 1; break
            if v: break
        o[k] = v
    return o
def clinical_f1(P, R):
    tp = fp = fn = 0
    for p, r in zip(P, R):
        a, b = assert_labels(p), assert_labels(r)
        for k in KW:
            tp += a[k] == 1 and b[k] == 1; fp += a[k] == 1 and b[k] == 0
            fn += a[k] == 0 and b[k] == 1
    pr = tp/max(tp+fp,1); rc = tp/max(tp+fn,1)
    return 2*pr*rc/max(pr+rc,1e-9)

@torch.no_grad()
def evaluate(dl, use_prompt=True, tag=''):
    model.eval(); P, R = [], []
    for x, lab, pid, pmask, txt in dl:
        x = x.to(DEV, non_blocking=True)
        a, b = (pid.to(DEV), pmask.to(DEV)) if use_prompt else (None, None)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            ids = model.generate(x, a, b, num_beams=1, max_length=GEN_MAX,
                                 min_length=GEN_MIN, no_repeat_ngram_size=3)
        P += TOK.batch_decode(ids, skip_special_tokens=True); R += list(txt)
    n = max(len(P), 1)
    rl = sum(_SC.score(r, p)['rougeL'].fmeasure for p, r in zip(P, R)) / n
    model.train()
    return dict(rougeL=rl, margin=rl - S4['baseline'], clinical_f1=clinical_f1(P, R),
                prior=sum(bool(PRIOR_RE.search(p)) for p in P)/n,
                gen_words=float(np.mean([len(p.split()) for p in P]))), P, R

m0, P0, R0 = evaluate(dl_va, use_prompt=True, tag='epoch0')
print('  epoch 0 (Stage 4 weights, prompt present but untrained):')
print('    ROUGE-L %.4f  clinF1 %.4f  prior %.4f  words %.1f'
      % (m0['rougeL'], m0['clinical_f1'], m0['prior'], m0['gen_words']))

---
# 5 · Training ★

Selection on **validation clinical F1** — that is the metric with the measured headroom, and the one that reflects whether the report is trustworthy. ROUGE-L is tracked but not optimised: a fixed string already scores 0.2769 on it.

In [ ]:
from tqdm.auto import tqdm
sfx  = '_smoke' if SMOKE_TEST else ''
LAST = CKPT_DIR / ('last' + sfx + '.pt'); BEST = CKPT_DIR / ('best' + sfx + '.pt')
start_ep, gstep, best_f1, hist = 0, 0, -1.0, []
if LAST.exists():
    r = torch.load(LAST, map_location='cpu', weights_only=False)
    model.load_state_dict(r['model']); opt.load_state_dict(r['opt'])
    sched.load_state_dict(r['sched'])
    start_ep, gstep, best_f1, hist = r['epoch']+1, r['gstep'], r['best_f1'], r['hist']
    print('  RESUMED from epoch', r['epoch'])
def atomic_save(o, p):
    t = p.with_suffix('.tmp'); torch.save(o, t); os.replace(t, p)

T0 = time.time()
for ep in range(start_ep, EPOCHS):
    model.train(); run = 0.0; n = 0
    bar = tqdm(dl_tr, desc='epoch %d/%d' % (ep+1, EPOCHS), leave=False)
    for x, lab, pid, pmask, _ in bar:
        x, lab = x.to(DEV, non_blocking=True), lab.to(DEV)
        pid, pmask = pid.to(DEV), pmask.to(DEV)
        with torch.autocast('cuda', dtype=torch.bfloat16):
            loss = model(x, lab, pid, pmask).loss
        assert torch.isfinite(loss), 'non-finite loss at step %d' % gstep
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        opt.step(); sched.step(); gstep += 1
        run += loss.item(); n += 1
        bar.set_postfix(loss='%.4f' % (run/n))
    mv, _, _ = evaluate(dl_va, use_prompt=True)
    mv.update(epoch=ep, loss=run/max(n,1)); hist.append(mv)
    print('  ep%d  loss %.4f | ROUGE-L %.4f  clinF1 %.4f  prior %.4f  words %.1f'
          % (ep+1, mv['loss'], mv['rougeL'], mv['clinical_f1'], mv['prior'], mv['gen_words']))
    if mv['prior'] > 0.005:
        print('       !! prior hallucination rose above 0. This must stay at 0.0000.')
    atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(),
                     epoch=ep, gstep=gstep, best_f1=best_f1, hist=hist), LAST)
    if mv['clinical_f1'] > best_f1:
        best_f1 = mv['clinical_f1']
        atomic_save(dict(model=model.state_dict(), epoch=ep, val=mv, hist=hist,
                         thresholds=THR), BEST)
        print('       ^ new best clinical F1 -> saved')
print('\n  trained in %.1f min' % ((time.time()-T0)/60))

---
# 6 · THE GATE — is the prompt actually being used? ★★

Generate the same images twice: once with the real prompt, once with the prompt **inverted** (positives and negatives swapped). If the reports are identical, the decoder is ignoring the prompt and the whole stage is a no-op — regardless of what the other metrics say.

In [ ]:
b = torch.load(BEST, map_location='cpu', weights_only=False)
model.load_state_dict(b['model']); model = model.to(DEV)
model.eval()
print('  loaded best epoch', b['epoch'] + 1)

# 16 images is too noisy to decide on. Sweep several batches. dl_te has
# shuffle=False, so row i of the loader is row off+i of PR['test'].
N_GATE = 96
tA, tB, off = [], [], 0
for x, lab, pid, pmask, txt in dl_te:
    n = len(x)
    inv = [s11.build_prompt(1.0 - PR['test'][off + i], THR) for i in range(n)]
    ipid, ipmask = s11.encode_prompts(inv, TOK, device='cpu')
    with torch.no_grad():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            gA = model.generate(x.to(DEV), pid.to(DEV), pmask.to(DEV), num_beams=1,
                                max_length=GEN_MAX, min_length=GEN_MIN, no_repeat_ngram_size=3)
            gB = model.generate(x.to(DEV), ipid.to(DEV), ipmask.to(DEV), num_beams=1,
                                max_length=GEN_MAX, min_length=GEN_MIN, no_repeat_ngram_size=3)
    tA += TOK.batch_decode(gA, skip_special_tokens=True)
    tB += TOK.batch_decode(gB, skip_special_tokens=True)
    off += n
    if len(tA) >= N_GATE:
        break

diff = sum(a != c for a, c in zip(tA, tB))
frac = diff / max(len(tA), 1)
print('  reports changed by inverting the prompt: %d / %d  (%.1f%%)' % (diff, len(tA), 100*frac))
print()
for i in range(len(tA)):
    if tA[i] != tB[i]:
        print('  REAL prompt    :', tA[i][:108])
        print('  INVERTED prompt:', tB[i][:108])
        break
print()
if frac > 0.5:
    print('  >>> PASS. The decoder is genuinely conditioning on the prompt.')
elif frac > 0.15:
    print('  >>> PARTIAL (%.1f%%). The prompt is being used but not dominant.' % (100*frac))
    print('      Treat any clinical F1 gain as real but modest. Report the sensitivity.')
else:
    print('  >>> FAIL. The decoder is ignoring the prompt -- this stage is a no-op.')
    print('      Do not report gains as coming from conditioning.')

---
# 7 · Final evaluation vs Stage 4

In [ ]:
mt, Pt, Rt = evaluate(dl_te, use_prompt=True)
print('=' * 84); print('  STAGE 4 vs STAGE 11  (same test set, n=%d)' % len(Rt)); print('=' * 84)
print('  %-30s %12s %12s %10s' % ('metric', 'Stage 4', 'Stage 11', 'change'))
print('  ' + '-' * 81)
for k, lab_ in (('clinical_f1', 'clinical F1  <- TARGET'), ('rougeL', 'ROUGE-L'),
                ('margin', 'margin over baseline'), ('prior', 'prior hallucination'),
                ('gen_words', 'mean words')):
    o, n_ = S4[k], mt[k]
    print('  %-30s %12.4f %12.4f %+10.4f' % (lab_, o, n_, n_ - o))
print('  ' + '-' * 81)
print()
print('  ceiling from the classifier prompt : 0.6535')
print('  captured                           : %.1f%% of the +0.0736 headroom'
      % (100 * (mt['clinical_f1'] - S4['clinical_f1']) / 0.0736))
print()
if mt['prior'] > 0.005:
    print('  !! WARNING: prior hallucination is no longer 0. That regression')
    print('     outweighs any clinical F1 gain -- report it.')
elif mt['clinical_f1'] > S4['clinical_f1'] + 0.01:
    print('  RESULT: clinical F1 improved and prior hallucination held at 0.')
else:
    print('  RESULT: no meaningful gain. Report honestly as a negative result.')
print('\n  per-epoch history')
for h in hist:
    print('   ep%d  loss %.4f  ROUGE-L %.4f  clinF1 %.4f  prior %.4f'
          % (h['epoch']+1, h['loss'], h['rougeL'], h['clinical_f1'], h['prior']))

---
# 8 · Save & verify both checkpoints intact

In [ ]:
from datetime import datetime
res = dict(stage=11, timestamp=datetime.now().isoformat(), smoke=SMOKE_TEST,
           epochs=EPOCHS, lr_vision=LR_VISION, lr_rest=LR_REST,
           prompt_dropout=PROMPT_DROP, thresholds=[float(t) for t in THR],
           best_epoch=int(b['epoch']), history=hist, test=mt, stage4_reference=S4,
           ceiling=0.6535, prompt_sensitivity=float(diff)/max(len(tA),1))
(OUT / ('stage11_results' + sfx + '.json')).write_text(
    json.dumps(res, indent=2, default=float), encoding='utf-8')
with open(OUT / ('stage11_samples' + sfx + '.txt'), 'w', encoding='utf-8') as f:
    for i in range(min(100, len(Pt))):
        f.write('--- [%d]\n  REF: %s\n  GEN: %s\n\n' % (i+1, ' '.join(Rt[i].split()),
                                                          ' '.join(Pt[i].split())))
print('  saved', OUT / ('stage11_results' + sfx + '.json'))
for n, p in (('stage4', S4_CKPT), ('stage5', S5_CKPT)):
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    assert h == SHA[n], n + ' WAS MODIFIED -- must never happen'
    print('  *** %s best.pt VERIFIED BYTE-IDENTICAL ***' % n)
if SMOKE_TEST:
    print('\n  SMOKE TEST ONLY. Set SMOKE_TEST = False, restart, re-run.')

---
# What this decides

| cell | question |
|---|---|
| §6 | is the decoder actually using the prompt, or ignoring it? |
| §7 | did clinical F1 improve, and did prior hallucination hold at 0? |
| §8 | are both original checkpoints intact? |

**Prior hallucination staying at 0.0000 is non-negotiable.** A clinical F1 gain bought by reintroducing fabricated references to prior scans is not a gain — it trades a measurable safety property for a metric, and §7 says so explicitly if it happens.